# Exercícios — Aula 5
## Limpeza e Tratamento de Dados com Pandas

**Aluno:** Rodrigo Gimenes Rodrigues  
**Matrícula:** CP3071782  
**Disciplina:** Processamento e Coleta de Dados


# Exercício 1: Caça aos NaNs

Carregue o dataset Titanic (`sns.load_dataset('titanic')`). Quantos NaNs existem na coluna 'age'? Preencha os NaNs com a mediana da idade agrupada por 'class' (1ª, 2ª, 3ª classe). *Dica: use `groupby` e `transform`.*

In [1]:
import seaborn as sns
import pandas as pd

df_titanic = sns.load_dataset('titanic')

nans_age = df_titanic['age'].isna().sum()
print('NaNs na coluna age:', nans_age)

# mediana de idade por classe, preenchendo só os NaNs
df_titanic['age'] = df_titanic['age'].fillna(
    df_titanic.groupby('class')['age'].transform('median')
)

print('NaNs depois do preenchimento:', df_titanic['age'].isna().sum())
print()
print('Mediana de idade por classe:')
print(df_titanic.groupby('class', observed=True)['age'].median())

NaNs na coluna age: 177
NaNs depois do preenchimento: 0

Mediana de idade por classe:
class
First     37.0
Second    29.0
Third     24.0
Name: age, dtype: float64


### Comentário
Antes do preenchimento, a coluna `age` tinha vários NaNs. Usei a mediana de cada classe (`First`, `Second`, `Third`) com `groupby` + `transform`, assim cada passageiro sem idade recebe a mediana da própria classe, e não a mediana geral do navio.

# Exercício 2: Padronização de E-mails

Crie um DataFrame com uma coluna de e-mails bagunçados (ex: " User@Email.com ", "USER@email.COM", "user@email.com"). Padronize todos para minúsculas e sem espaços. Verifique quantos e-mails únicos existem após a limpeza.

In [2]:
df_emails = pd.DataFrame({
    'email': [
        ' User@Email.com ',
        'USER@email.COM',
        'user@email.com',
        '  Ana.Silva@Gmail.com',
        'ana.silva@gmail.com ',
        'teste@TEST.COM'
    ]
})

print('Antes:')
print(df_emails)
print('Únicos antes:', df_emails['email'].nunique())

df_emails['email'] = df_emails['email'].str.lower().str.strip()

print()
print('Depois:')
print(df_emails)
print('Únicos depois:', df_emails['email'].nunique())

Antes:
                   email
0        User@Email.com 
1         USER@email.COM
2         user@email.com
3    Ana.Silva@Gmail.com
4   ana.silva@gmail.com 
5         teste@TEST.COM
Únicos antes: 6

Depois:
                 email
0       user@email.com
1       user@email.com
2       user@email.com
3  ana.silva@gmail.com
4  ana.silva@gmail.com
5       teste@test.com
Únicos depois: 3


### Comentário
Depois de `lower()` + `strip()`, e-mails que pareciam diferentes viram o mesmo valor. Por isso o número de e-mails únicos cai.

# Exercício 3: Conversão de Moeda e Data

Crie um DataFrame com vendas contendo: "Data" (formatos mistos: "01/01/2023", "2023-01-02") e "Valor" (formato "R$ 1.500,50"). Converta "Data" para datetime e "Valor" para float. Calcule o total vendido por mês.

In [3]:
df_vendas = pd.DataFrame({
    'Data': ['01/01/2023', '2023-01-02', '15/02/2023', '2023-02-20', '03/03/2023'],
    'Valor': ['R$ 1.500,50', 'R$ 200,00', 'R$ 3.250,90', 'R$ 890,10', 'R$ 1.100,00']
})

print('Antes:')
print(df_vendas)
print(df_vendas.dtypes)

df_vendas['Data'] = pd.to_datetime(df_vendas['Data'], format='mixed', dayfirst=True)

df_vendas['Valor'] = (
    df_vendas['Valor']
    .str.replace('R$', '', regex=False)
    .str.strip()
    .str.replace('.', '', regex=False)
    .str.replace(',', '.')
)
df_vendas['Valor'] = pd.to_numeric(df_vendas['Valor'])

print()
print('Depois:')
print(df_vendas)
print(df_vendas.dtypes)

total_mes = df_vendas.groupby(df_vendas['Data'].dt.to_period('M'))['Valor'].sum()
print()
print('Total vendido por mês:')
print(total_mes)

Antes:
         Data        Valor
0  01/01/2023  R$ 1.500,50
1  2023-01-02    R$ 200,00
2  15/02/2023  R$ 3.250,90
3  2023-02-20    R$ 890,10
4  03/03/2023  R$ 1.100,00
Data     str
Valor    str
dtype: object

Depois:
        Data   Valor
0 2023-01-01  1500.5
1 2023-02-01   200.0
2 2023-02-15  3250.9
3 2023-02-20   890.1
4 2023-03-03  1100.0
Data     datetime64[us]
Valor           float64
dtype: object

Total vendido por mês:
Data
2023-01    1500.5
2023-02    4341.0
2023-03    1100.0
Freq: M, Name: Valor, dtype: float64


### Comentário
A data mista foi resolvida com `to_datetime(..., format='mixed', dayfirst=True)`. No valor em reais, tirei `R$`, removi o ponto de milhar e troquei a vírgula por ponto antes do `to_numeric`. Com isso dá para agrupar por mês sem erro.